# AVA: genuine CURE, K-Means, and a separate Ridge baseline

Upload this notebook to **Google Colab**. Choose a **CPU, high-RAM runtime**
(Runtime > Change runtime type), mount Drive, and edit `DATA_PATH` below.
A GPU/TPU does not accelerate these scikit-learn / pyclustering CPU estimators.
For ~250,000 images, prefer **at least 25-32 GB RAM** as a starting point, not a
guaranteed requirement; monitor actual memory and runtime. Keep searches sequential.

**Run sections 1-4 first**, inspect the CURE benchmark, then enable the clustering
search in section 5. Section 6 runs Ridge independently, even if you skip clustering.

* 3 shuffled outer folds estimate generalization; 3 inner folds select parameters
  by **MSE**. The same image splits are used for every method.
* K-Means fits every training image. **Genuine CURE** uses pyclustering's C++
  representative-distance merging algorithm on a reproducible, training-only
  sample of **10,000 images** with one fixed, reproducible sampling seed.
  Sampling seeds are not tuned. This is not Ward clustering.
* Both cluster regressors predict the mean rating of **all training images**
  assigned to the cluster, not just the sampled images.
* Ridge is a separate, standardized full-feature linear model with inner-CV
  regularization tuning. Clustering uses standardized features plus 20-component
  PCA; Ridge retains all features. The evaluated difference includes this
  preprocessing choice, not only the estimator.
* Ratings never enter clustering, scaling, or PCA. Every preprocessing fit is
  restricted to its training fold. Only frozen CLIP and structural features enter X.

“Best” means best **among the configured candidates**, not a global optimum.
Sampling CURE changes its fitted structure; results describe sampled CURE,
not full-data CURE. Random image-level CV may be optimistic for near-duplicates
or shared photographers. Use grouped splits if such identifiers are available.

All CSVs, configuration, timings, plots, and checkpoints go into a **named Drive
run folder**. Completed fits resume automatically from that folder. Outer-fold
models and predictions are retained. No existing notebook is modified.

## 1. Dependencies and Google Drive

This pins the older pyclustering release explicitly. Its compiled backend is checked
below: **do not silently fall back to its much slower Python implementation**.
NumPy 2 is used with the C++ CURE backend; no NumPy compatibility monkey patches.
If Colab reports already-imported packages changed, restart the session after this cell.

In [ ]:
%pip install -q "numpy>=2,<3" "scipy>=1.13,<2" "pandas>=2.2,<3" "pyarrow>=18" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" "pyclustering==0.10.1.2"

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import platform
import re
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pyclustering.cluster.cure import cure
from pyclustering.core.wrapper import ccore_library
from scipy.spatial.distance import cdist
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, ParameterGrid
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils.validation import check_array, check_is_fitted, check_X_y
from threadpoolctl import threadpool_limits

try:
    from google.colab import drive
except ImportError:
    print("Not in Colab: configure a local DATA_PATH and OUTPUT_ROOT.")
else:
    drive.mount("/content/drive")


def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w") as stream:
        json.dump(payload, stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def atomic_model(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(payload, temporary, compress=3)
    with temporary.open("rb") as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, path)


def atomic_csv(frame, path, index=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary, index=index)
    with temporary.open("rb") as stream:
        os.fsync(stream.fileno())
    os.replace(temporary, path)

## 2. Paths, candidate grids, and data integrity

Set `DATA_PATH` to **your full-data Parquet on Drive**, not the older local 10% file.
The notebook checks the actual row count and feature widths; it does not assume
that a file with a particular name contains 250,000 images.

**To resume after a disconnect:** mount Drive, use the same `DATA_PATH`,
`OUTPUT_ROOT`, `RUN_NAME`, and candidate settings, then rerun the notebook from
the top (including enabling section 5). Completed scores/models are restored
and skipped. To start a different experiment, change `RUN_NAME`.
The configuration and a hash of ordered feature/target/ID data must match.
Recorded Python/package versions must also match; if Colab's environment changed,
install the versions in `config.json` before resuming. Do not run two sessions
against the same run folder concurrently. Only load your own trusted checkpoints.
Old timestamped CSV-only runs cannot be resumed by this checkpoint format.

`k` is the number of clusters (more clusters means smaller average groups).
`n_reps` is the maximum representatives per CURE cluster.
`shrink` is the fraction moved toward the centroid: 0 = unchanged, 1 = centroid.
`CURE_SAMPLE_CAP` is a computational budget, fixed across candidates, not a tuned
parameter here. At 10,000 sampled points and k=4,200, the average cluster has only
2.4 sample points; representative count is capped by actual cluster membership.
`CURE_SAMPLE_SEEDS` contains only `[11]`: a fixed seed, not a seed search.
It is combined with each fold's seed,
so each training fold draws its own reproducible sample. All parameter candidates
within that fold share the same sampled images.

In [ ]:
DATA_PATH = Path("/content/drive/MyDrive/ava_cleaned_features.parquet")  # EDIT THIS.
OUTPUT_ROOT = Path("/content/drive/MyDrive/ava_nested_cv_results")
RUN_NAME = "ava_3x3_cure10k_single_sample_v1"  # Keep unchanged to resume.

SEED = 42
OUTER_FOLDS = 3
INNER_FOLDS = 3
PCA_COMPONENTS = 20
CPU_THREADS = min(4, os.cpu_count() or 1)
KMEANS_N_INIT = 5
K_VALUES = [48, 96, 192, 396, 796, 1200, 1800, 2600, 3400, 4200]
CURE_K_VALUES = [396, 796, 1800, 2600, 4200]
CURE_REP_VALUES = [3, 5]
CURE_SHRINK_VALUES = [0.1, 0.2, 0.5]
CURE_SAMPLE_CAP = 10_000
CURE_SAMPLE_SEEDS = [11]  # One fixed sample seed; no three-sample search.
RIDGE_ALPHAS = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0]

# Enable only after reviewing section 4. Ridge can run independently.
RUN_CLUSTER_CV = False
RUN_FINAL_CLUSTER_MODELS = False  # Extra full-data 3-fold tuning and refits.
RUN_FINAL_RIDGE_MODEL = True

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Edit DATA_PATH: no Parquet at {DATA_PATH}")
metadata = pq.ParquetFile(DATA_PATH).metadata
print(f"Parquet contains {metadata.num_rows:,} rows: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)
required = {"image_id", "mean_score", "clip_embedding", "structural_features"}
if not required.issubset(df.columns):
    raise ValueError(f"Missing columns: {sorted(required - set(df.columns))}")
if df.empty or df[list(required)].isna().any().any():
    raise ValueError("Empty dataset or missing IDs, scores, or features.")
image_ids = df["image_id"].astype(str).to_numpy()
if pd.Series(image_ids).duplicated().any():
    raise ValueError("Duplicate image IDs: deduplicate or use grouped CV.")
y = df["mean_score"].to_numpy(dtype=np.float64)
if not np.isfinite(y).all() or ((y < 1) | (y > 10)).any():
    raise ValueError("Mean ratings must be finite and on the 1-10 scale.")
X_clip = np.stack(df["clip_embedding"]).astype(np.float32)
X_struct = np.stack(df["structural_features"]).astype(np.float32)
if X_clip.shape != (len(y), 512) or X_struct.shape != (len(y), 12):
    raise ValueError("Expected 512 CLIP and 12 structural features per image.")
X = np.hstack([X_clip, X_struct])
if not np.isfinite(X).all():
    raise ValueError("Features contain NaN or infinity.")
if "rating_counts" in df:
    counts = np.stack(df["rating_counts"]).astype(np.float64)
    if (
        counts.shape != (len(y), 10)
        or not np.isfinite(counts).all()
        or (counts < 0).any()
        or not np.equal(counts, np.floor(counts)).all()
    ):
        raise ValueError("Expected ten non-negative integer vote counts per image.")
    totals = counts.sum(axis=1)
    if (totals <= 0).any():
        raise ValueError("Every image needs positive rating support.")
    recomputed = (counts @ np.arange(1, 11)) / totals
    if not np.allclose(recomputed, y, atol=1e-6, rtol=0):
        raise ValueError("Stored mean ratings disagree with vote counts.")
    if "total_votes" in df and not np.array_equal(totals, df["total_votes"]):
        raise ValueError("Stored vote totals disagree with counts.")
    del counts, totals, recomputed
del df, X_clip, X_struct

if re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]*", RUN_NAME) is None:
    raise ValueError(
        "RUN_NAME must contain only letters, digits, underscores, and hyphens."
    )
RUN_DIR = OUTPUT_ROOT / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
data_hash = hashlib.sha256()
data_hash.update(memoryview(X))
data_hash.update(memoryview(y))
data_hash.update(json.dumps(image_ids.tolist(), ensure_ascii=True).encode())
config = {
    "checkpoint_version": 1,
    "data_sha256": data_hash.hexdigest(),
    "data_path": str(DATA_PATH),
    "n_images": len(y),
    "n_features": X.shape[1],
    "seed": SEED,
    "outer_folds": OUTER_FOLDS,
    "inner_folds": INNER_FOLDS,
    "pca_components": PCA_COMPONENTS,
    "cpu_threads": CPU_THREADS,
    "kmeans_n_init": KMEANS_N_INIT,
    "k_values": K_VALUES,
    "cure_k_values": CURE_K_VALUES,
    "cure_rep_values": CURE_REP_VALUES,
    "cure_shrink_values": CURE_SHRINK_VALUES,
    "cure_sample_cap": CURE_SAMPLE_CAP,
    "cure_sample_seeds": CURE_SAMPLE_SEEDS,
    "ridge_alphas": RIDGE_ALPHAS,
    "python": platform.python_version(),
    "versions": {
        name: importlib.metadata.version(name)
        for name in [
            "numpy",
            "scipy",
            "pandas",
            "pyarrow",
            "scikit-learn",
            "pyclustering",
        ]
    },
}
config_path = RUN_DIR / "config.json"
if config_path.is_file():
    stored_config = json.loads(config_path.read_text())
    if stored_config != config:
        changed = sorted(
            key
            for key in set(config) | set(stored_config)
            if config.get(key) != stored_config.get(key)
        )
        raise ValueError(
            f"Cannot resume: changed configuration/data fields: {changed}. "
            "Restore the original settings/packages or choose a new RUN_NAME."
        )
    print("Resuming existing run:", RUN_DIR)
else:
    if any(RUN_DIR.iterdir()):
        raise ValueError(
            "Run folder contains files but no config.json; choose a new RUN_NAME."
        )
    atomic_json(config_path, config)
    print("Starting new run:", RUN_DIR)
CONFIG_HASH = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
print(f"X: {X.shape}, {X.nbytes / 2**30:.2f} GiB; outputs: {RUN_DIR}")
print(
    "Keep RUN_NAME and settings unchanged to resume; use a new name for changed grids."
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y, bins=np.linspace(1, 10, 37), edgecolor="white")
ax.set(
    xlabel="Mean aesthetic rating",
    ylabel="Images",
    title="Full-file target distribution",
)
fig.tight_layout()
fig.savefig(RUN_DIR / "rating_distribution.png", dpi=160)
plt.show()
display(pd.Series(y, name="mean_score").describe(percentiles=[0.01, 0.05, 0.95, 0.99]))

## 3. Reusable estimators and identical nested splits

CURE representatives determine the merges **inside the library**. Training and
prediction then use exact nearest-representative assignment in bounded distance
blocks, avoiding a full images-by-representatives matrix. A CURE cluster with no
assigned training images is an explicit error, not a fallback mean.

Preprocessing is fitted once per training split and reused across candidates:
this avoids repeatedly fitting identical scalers/PCA without leaking validation data.
Each candidate is selected using the mean of its three inner-fold MSEs.

In [ ]:
def require_cure_core():
    try:
        available = ccore_library.workable()
    except OSError as exc:
        raise RuntimeError(
            "CURE C++ backend failed to load. Use standard Linux x86-64 Colab; "
            "do not run the 10,000-point Python fallback."
        ) from exc
    if not available:
        raise RuntimeError(
            "CURE C++ backend unavailable; stop and fix the installation."
        )


class ClusterMeanRegressor(RegressorMixin, BaseEstimator):
    def __init__(
        self,
        method="kmeans",
        n_clusters=396,
        n_reps=5,
        shrink=0.2,
        sample_cap=10_000,
        sample_seed=0,
        n_init=5,
        random_state=42,
    ):
        self.method = method
        self.n_clusters = n_clusters
        self.n_reps = n_reps
        self.shrink = shrink
        self.sample_cap = sample_cap
        self.sample_seed = sample_seed
        self.n_init = n_init
        self.random_state = random_state

    def fit(self, X, y):
        X, y = check_X_y(X, y, y_numeric=True)
        self.n_features_in_ = X.shape[1]
        if not 1 <= self.n_clusters <= len(X):
            raise ValueError("n_clusters must be between 1 and the training row count.")
        started = time.perf_counter()
        if self.method == "kmeans":
            self.clusterer_ = KMeans(
                n_clusters=self.n_clusters,
                n_init=self.n_init,
                algorithm="lloyd",
                random_state=self.random_state,
            ).fit(X)
            self.sample_size_ = len(X)
        elif self.method == "cure":
            require_cure_core()
            self.sample_size_ = self.sample_cap
            if (
                self.sample_size_ > len(X)
                or self.sample_size_ <= self.n_clusters
                or self.n_reps < 1
                or not 0 <= self.shrink <= 1
            ):
                raise ValueError(
                    "CURE needs k < sample_size <= training_size, n_reps >= 1, and shrink in [0,1]."
                )
            sampling_seed = np.random.SeedSequence(
                [self.random_state, self.sample_seed]
            )
            self.sample_indices_ = np.random.default_rng(sampling_seed).choice(
                len(X), self.sample_size_, replace=False
            )
            fitted = cure(
                X[self.sample_indices_].astype(np.float64).tolist(),
                self.n_clusters,
                self.n_reps,
                self.shrink,
                ccore=True,
            ).process()
            groups = fitted.get_representors()
            if len(groups) != self.n_clusters or any(
                len(group) == 0 for group in groups
            ):
                raise RuntimeError("CURE returned an invalid representative structure.")
            self.representatives_ = np.vstack(groups).astype(np.float64)
            self.representative_labels_ = np.repeat(
                np.arange(self.n_clusters), [len(group) for group in groups]
            )
            if not np.isfinite(self.representatives_).all():
                raise RuntimeError("CURE returned non-finite representatives.")
        else:
            raise ValueError(f"Unknown clustering method: {self.method}")
        self.clustering_seconds_ = time.perf_counter() - started
        labels = self._assign(X)
        self.cluster_counts_ = np.bincount(labels, minlength=self.n_clusters)
        if (self.cluster_counts_ == 0).any():
            raise ValueError(
                "A cluster has no nearest-assigned training images. "
                "Use less shrinkage, fewer clusters, or a larger CURE sample."
            )
        self.cluster_means_ = (
            np.bincount(labels, weights=y, minlength=self.n_clusters)
            / self.cluster_counts_
        )
        self.fit_seconds_ = time.perf_counter() - started
        return self

    def _assign(self, X):
        if self.method == "kmeans":
            return self.clusterer_.predict(X)
        labels = np.empty(len(X), dtype=np.int64)
        for start in range(0, len(X), 512):
            block = X[start : start + 512]
            best_distance = np.full(len(block), np.inf)
            best_index = np.zeros(len(block), dtype=np.int64)
            for rep_start in range(0, len(self.representatives_), 2048):
                distances = cdist(
                    block,
                    self.representatives_[rep_start : rep_start + 2048],
                    metric="sqeuclidean",
                )
                local_index = distances.argmin(axis=1)
                local_distance = distances[np.arange(len(block)), local_index]
                improved = local_distance < best_distance
                best_distance[improved] = local_distance[improved]
                best_index[improved] = rep_start + local_index[improved]
            labels[start : start + len(block)] = self.representative_labels_[best_index]
        return labels

    def predict(self, X):
        check_is_fitted(self, "cluster_means_")
        X = check_array(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("Prediction feature width differs from training.")
        return self.cluster_means_[self._assign(X)]


def cluster_preprocessor(seed):
    return make_pipeline(
        StandardScaler(),
        PCA(n_components=PCA_COMPONENTS, svd_solver="randomized", random_state=seed),
    )


def cluster_estimator(method, params, seed):
    return ClusterMeanRegressor(
        method=method,
        sample_cap=CURE_SAMPLE_CAP,
        n_init=KMEANS_N_INIT,
        random_state=seed,
        **params,
    )


def metrics(actual, prediction):
    return {
        "mse": mean_squared_error(actual, prediction),
        "rmse": np.sqrt(mean_squared_error(actual, prediction)),
        "mae": mean_absolute_error(actual, prediction),
        "r2": r2_score(actual, prediction),
    }


def method_slug(name):
    return name.lower().replace("-", "_").replace(" ", "_")


def read_json_checkpoint(path):
    if not path.is_file():
        return None
    payload = json.loads(path.read_text())
    if payload["config_hash"] != CONFIG_HASH:
        raise ValueError(f"Checkpoint belongs to a different configuration: {path}")
    return payload["data"]


def write_json_checkpoint(path, data):
    atomic_json(path, {"config_hash": CONFIG_HASH, "data": data})


def read_model_checkpoint(path):
    if not path.is_file():
        return None
    payload = joblib.load(path)
    if payload["config_hash"] != CONFIG_HASH:
        raise ValueError(
            f"Model checkpoint belongs to a different configuration: {path}"
        )
    return payload


def candidate_path(context, name, candidate, inner_fold):
    return (
        CHECKPOINT_DIR
        / "inner"
        / context
        / (f"{method_slug(name)}_candidate_{candidate}_inner_{inner_fold}.json")
    )


def remember_tuning(row):
    key = (row["context"], row["method"], row["candidate"], row["inner_fold"])
    if key not in seen_tuning_keys:
        tuning_records.append(row)
        seen_tuning_keys.add(key)


def cached_candidate(path, expected):
    row = read_json_checkpoint(path)
    if row is not None:
        if any(row.get(key) != value for key, value in expected.items()):
            raise ValueError(
                f"Candidate checkpoint does not match its expected parameters: {path}"
            )
        if not np.isfinite(row["mse"]):
            raise ValueError(f"Non-finite checkpoint MSE: {path}")
        remember_tuning(row)
    return row


def outer_model_path(fold, name):
    return CHECKPOINT_DIR / "models" / f"outer_{fold}_{method_slug(name)}.joblib"


def save_progress():
    atomic_csv(pd.DataFrame(tuning_records), RUN_DIR / "tuning_progress.csv")
    if fold_records:
        atomic_csv(pd.DataFrame(fold_records), RUN_DIR / "fold_scores.csv")
    if oof_predictions:
        atomic_csv(
            pd.DataFrame(
                {
                    "image_id": image_ids,
                    "mean_score": y,
                    "outer_fold": outer_fold,
                    **oof_predictions,
                }
            ),
            RUN_DIR / "oof_progress.csv",
        )


def apply_outer_checkpoint(payload, fold, name):
    _, test = outer_splits[fold - 1]
    if (
        payload["fold"] != fold
        or payload["method"] != name
        or not np.array_equal(payload["test_indices"], test)
    ):
        raise ValueError("Outer checkpoint fold or test indices do not match this run.")
    for method, prediction in payload["predictions"].items():
        if prediction.shape != (len(test),) or not np.isfinite(prediction).all():
            raise ValueError("Invalid predictions in outer checkpoint.")
        oof_predictions.setdefault(method, np.full(len(y), np.nan))[test] = prediction
    for record in payload["records"]:
        if record["fold"] != fold:
            raise ValueError("Wrong fold in outer checkpoint metrics.")
        fold_records[:] = [
            old
            for old in fold_records
            if (old["fold"], old["method"]) != (fold, record["method"])
        ]
        fold_records.append(record)
    if name == "CURE":
        atomic_csv(
            pd.DataFrame(payload["sample_data"]),
            RUN_DIR / f"cure_selected_sample_outer_{fold}.csv",
        )
    completed_outer.add((fold, name))


def restore_outer(fold, name):
    if (fold, name) not in completed_outer:
        payload = read_model_checkpoint(outer_model_path(fold, name))
        if payload is not None:
            apply_outer_checkpoint(payload, fold, name)
            print(f"Restored outer {fold}: {name}", flush=True)
    return (fold, name) in completed_outer


def commit_outer(payload):
    payload["config_hash"] = CONFIG_HASH
    atomic_model(outer_model_path(payload["fold"], payload["method"]), payload)
    apply_outer_checkpoint(payload, payload["fold"], payload["method"])
    save_progress()

In [ ]:
if (
    len(CURE_SAMPLE_SEEDS) != 1
    or not isinstance(CURE_SAMPLE_SEEDS[0], int)
    or CURE_SAMPLE_SEEDS[0] < 0
):
    raise ValueError("Configure exactly one non-negative CURE sample seed.")
cluster_grids = {
    "K-Means": list(ParameterGrid({"n_clusters": K_VALUES})),
    "CURE": list(
        ParameterGrid(
            {
                "n_clusters": CURE_K_VALUES,
                "n_reps": CURE_REP_VALUES,
                "shrink": CURE_SHRINK_VALUES,
                "sample_seed": CURE_SAMPLE_SEEDS,
            }
        )
    ),
}
outer_splits = list(
    KFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=SEED).split(X)
)
inner_splits = {
    fold: list(
        KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED + fold).split(train)
    )
    for fold, (train, _) in enumerate(outer_splits, 1)
}
smallest_train = min(
    len(inner_train) for splits in inner_splits.values() for inner_train, _ in splits
)
if max(K_VALUES) > smallest_train:
    raise ValueError("K-Means candidates exceed an inner training fold's size.")
if CURE_SAMPLE_CAP > smallest_train:
    raise ValueError(
        "Every inner training fold must contain the full fixed-size CURE sample."
    )
if max(CURE_K_VALUES) >= CURE_SAMPLE_CAP:
    raise ValueError(
        "CURE sample size must exceed the largest cluster-count candidate."
    )
if PCA_COMPONENTS > min(smallest_train, X.shape[1]):
    raise ValueError(
        "PCA dimension exceeds an inner training fold's available dimensions."
    )
outer_fold = np.zeros(len(y), dtype=int)
for fold, (_, test) in enumerate(outer_splits, 1):
    outer_fold[test] = fold
atomic_csv(
    pd.DataFrame({"image_id": image_ids, "outer_fold": outer_fold}),
    RUN_DIR / "outer_split_assignments.csv",
)
for name, grid in cluster_grids.items():
    print(
        f"{name}: {len(grid)} candidates, "
        f"{len(grid) * OUTER_FOLDS * INNER_FOLDS} inner fits + {OUTER_FOLDS} outer refits."
    )
print(f"Smallest inner training fold: {smallest_train:,} images.")
print(
    f"CURE sample size: {CURE_SAMPLE_CAP:,}; fixed sample seed: {CURE_SAMPLE_SEEDS[0]}; "
    f"CPU threads: {CPU_THREADS}."
)

oof_predictions = {}
fold_records = []
tuning_records = []
seen_tuning_keys = set()
completed_outer = set()
for path in sorted((CHECKPOINT_DIR / "inner").glob("*/*.json")):
    row = read_json_checkpoint(path)
    if not np.isfinite(row["mse"]):
        raise ValueError(f"Non-finite cached MSE: {path}")
    remember_tuning(row)
for fold in range(1, OUTER_FOLDS + 1):
    for name in ["K-Means", "CURE", "Ridge"]:
        restore_outer(fold, name)
save_progress()
print(
    f"Restored {len(tuning_records)} inner fit scores and {len(completed_outer)} outer models."
)

## 4. Benchmark genuine CURE before committing to the grid

This uses **only the first outer training fold**. It does not inspect outer-test
ratings. One reproducible **10,000-image sample** measures CURE hierarchy building
**and assignment of all available training images** separately. A single representative parameter setting
cannot predict timings for every candidate; reported search hours are a rough
budgeting scenario, not a guarantee or linear scaling model. No peak-RAM estimate is
claimed; inspect Colab's RAM monitor during these fits.

The default benchmark uses k=396, five representatives and shrink=0.2.
If a fit is too slow, stop the cell and reduce the grid before starting
a new run from section 2. Higher k means fewer merges but more representative
comparisons during assignment, so total time need not improve.
The completed benchmark is checkpointed too; restarts skip it.

In [ ]:
benchmark_rows = []
pending_benchmarks = []
for sample_seed in CURE_SAMPLE_SEEDS:
    path = CHECKPOINT_DIR / "benchmark" / f"sample_{sample_seed}.json"
    cached = read_json_checkpoint(path)
    if cached is None:
        pending_benchmarks.append((sample_seed, path))
    else:
        benchmark_rows.append(cached)
train, _ = outer_splits[0]
if pending_benchmarks:
    require_cure_core()
    with threadpool_limits(limits=CPU_THREADS):
        started = time.perf_counter()
        benchmark_preprocessor = cluster_preprocessor(SEED)
        benchmark_X = benchmark_preprocessor.fit_transform(X[train])
        preprocessing_seconds = time.perf_counter() - started
        for sample_seed, path in pending_benchmarks:
            model = ClusterMeanRegressor(
                method="cure",
                n_clusters=min(CURE_K_VALUES),
                n_reps=5,
                shrink=0.2,
                sample_cap=CURE_SAMPLE_CAP,
                sample_seed=sample_seed,
                random_state=SEED + 100,
            ).fit(benchmark_X, y[train])
            row = {
                "sample_size": CURE_SAMPLE_CAP,
                "sample_seed": sample_seed,
                "training_rows": len(train),
                "preprocessing_seconds": preprocessing_seconds,
                "hierarchy_seconds": model.clustering_seconds_,
                "assignment_and_rating_seconds": model.fit_seconds_
                - model.clustering_seconds_,
                "total_fit_seconds": model.fit_seconds_,
                "representatives": len(model.representatives_),
            }
            write_json_checkpoint(path, row)
            benchmark_rows.append(row)
            print(row, flush=True)
        del model, benchmark_X, benchmark_preprocessor
benchmark = pd.DataFrame(benchmark_rows).sort_values("sample_seed")
atomic_csv(benchmark, RUN_DIR / "cure_benchmark.csv")
display(benchmark)
seconds = benchmark["total_fit_seconds"].mean()
fits = len(cluster_grids["CURE"]) * OUTER_FOLDS * INNER_FOLDS + OUTER_FOLDS
print(
    f"If every CURE fit took the benchmark mean of {seconds:.1f}s, {fits} fits would take "
    f"{seconds * fits / 3600:.1f} hours, excluding preprocessing, validation "
    "prediction, K-Means, Ridge, and optional final tuning. Actual candidate timings will differ."
)

## 5. Clustering: 3 outer x 3 inner nested CV

Set `RUN_CLUSTER_CV = True` **in the next cell after reviewing the benchmark**.
Every completed inner candidate's MSE is saved immediately in an atomic JSON
checkpoint on Drive. Restarts skip those fits. Each outer refit is saved as one
joblib checkpoint containing **the fitted pipeline, held-out predictions, metrics,
and selected CURE sample IDs**. Progress CSVs are rebuilt from checkpoints.
An interrupted individual fit must restart; completed fits do not repeat.
Only inner scores are retained, not hundreds of unused inner candidate models.

All CURE candidates in an inner fold use **one fixed sample of exactly 10,000
training images**. There is no three-sample search. Inner CV selects only k,
representative count, and shrinkage. The fixed seed generates a fresh sample for
each outer refit; the sampled image IDs are saved.
With the default grid, this is **270 inner CURE fits plus 3 outer refits**.

Candidates are intentionally searched separately for K-Means and CURE. CURE's
smaller k grid limits runtime; expand it explicitly if you want identical grids.
No candidate failures are silently ignored. Tie-breaking follows the deterministic
candidate ordering. Three-fold SD is descriptive, not a confidence interval.

In [ ]:
# Change to True here only after reviewing the benchmark.
RUN_CLUSTER_CV = False


def tune_clusters(X_train, y_train, splits, seed, context, methods=None):
    methods = list(cluster_grids) if methods is None else methods
    scores = {
        name: np.empty((len(grid), len(splits)))
        for name, grid in cluster_grids.items()
        if name in methods
    }
    for inner_fold, (train, valid) in enumerate(splits, 1):
        pending = []
        for name in methods:
            for candidate, params in enumerate(cluster_grids[name]):
                expected = {
                    "context": context,
                    "inner_fold": inner_fold,
                    "method": name,
                    "candidate": candidate,
                    **params,
                }
                path = candidate_path(context, name, candidate, inner_fold)
                cached = cached_candidate(path, expected)
                if cached is None:
                    pending.append((name, candidate, params, path, expected))
                else:
                    scores[name][candidate, inner_fold - 1] = cached["mse"]
        if not pending:
            print(
                f"{context}, inner {inner_fold}: all requested cluster scores restored.",
                flush=True,
            )
            continue
        preprocessor = cluster_preprocessor(seed + inner_fold)
        train_features = preprocessor.fit_transform(X_train[train])
        valid_features = preprocessor.transform(X_train[valid])
        for name, candidate, params, path, expected in pending:
            method = "kmeans" if name == "K-Means" else "cure"
            started = time.perf_counter()
            model = cluster_estimator(method, params, seed + inner_fold).fit(
                train_features, y_train[train]
            )
            mse = mean_squared_error(y_train[valid], model.predict(valid_features))
            row = {**expected, "mse": mse, "seconds": time.perf_counter() - started}
            write_json_checkpoint(path, row)
            remember_tuning(row)
            scores[name][candidate, inner_fold - 1] = mse
            print(
                f"{context}, inner {inner_fold}/{len(splits)}, {name} "
                f"{candidate + 1}/{len(cluster_grids[name])}, {params}: "
                f"MSE={mse:.5f} [checkpoint saved]",
                flush=True,
            )
        atomic_csv(pd.DataFrame(tuning_records), RUN_DIR / "tuning_progress.csv")
    result = {}
    for name, matrix in scores.items():
        means = matrix.mean(axis=1)
        best = int(means.argmin())
        result[name] = (cluster_grids[name][best], float(means[best]))
    return result


if RUN_CLUSTER_CV:
    for name in cluster_grids:
        oof_predictions.setdefault(name, np.full(len(y), np.nan))
    with threadpool_limits(limits=CPU_THREADS):
        for fold, (train, test) in enumerate(outer_splits, 1):
            pending_methods = [
                name for name in cluster_grids if not restore_outer(fold, name)
            ]
            if not pending_methods:
                print(
                    f"Outer {fold}: clustering models already complete; skipped.",
                    flush=True,
                )
                continue
            X_train, y_train = X[train], y[train]
            selected = tune_clusters(
                X_train,
                y_train,
                inner_splits[fold],
                SEED + fold * 100,
                context=f"outer_{fold}",
                methods=pending_methods,
            )
            preprocessor = cluster_preprocessor(SEED + fold * 100)
            train_features = preprocessor.fit_transform(X_train)
            test_features = preprocessor.transform(X[test])
            for name in pending_methods:
                method = "kmeans" if name == "K-Means" else "cure"
                params, inner_mse = selected[name]
                model = cluster_estimator(method, params, SEED + fold * 100).fit(
                    train_features, y_train
                )
                prediction = model.predict(test_features)
                record = {
                    "fold": fold,
                    "method": name,
                    "best_params": json.dumps(params),
                    "inner_mse": inner_mse,
                    "n_train": len(train),
                    "n_test": len(test),
                    "fit_seconds": model.fit_seconds_,
                    "min_cluster_count": int(model.cluster_counts_.min()),
                    "median_cluster_count": float(np.median(model.cluster_counts_)),
                    **metrics(y[test], prediction),
                }
                payload = {
                    "fold": fold,
                    "method": name,
                    "model": make_pipeline(preprocessor, model),
                    "test_indices": test,
                    "predictions": {name: prediction},
                    "records": [record],
                }
                if name == "CURE":
                    payload["sample_data"] = {
                        "image_id": image_ids[train[model.sample_indices_]],
                        "dataset_row": train[model.sample_indices_],
                        "sample_seed": np.full(model.sample_size_, model.sample_seed),
                    }
                commit_outer(payload)
                print(
                    f"Outer {fold}: {name}, best={params}, "
                    f"held-out MSE={record['mse']:.5f} [model checkpoint saved]",
                    flush=True,
                )
            del model, preprocessor, train_features, test_features, X_train, y_train
else:
    print(
        "Clustering search skipped. Set RUN_CLUSTER_CV=True above after benchmarking."
    )

## 6. Separate Ridge regression and mean-only baseline

Ridge runs independently of the clustering search, on the **same saved outer folds**.
For each inner fold, the scaler learns only from its training rows. Alpha is selected
by validation MSE; the winning scaler/Ridge model is refitted on the entire outer
training fold and evaluated once on the outer test fold.

Ridge uses all 524 standardized features, **no PCA and no rating clipping**.
The mean-only baseline predicts the outer-training mean. Its purpose is to show
whether either method improves over ignoring image features altogether.

In [ ]:
oof_predictions.setdefault("Ridge", np.full(len(y), np.nan))
oof_predictions.setdefault("Mean baseline", np.full(len(y), np.nan))


def tune_ridge(X_train, y_train, splits, context):
    scores = np.empty((len(RIDGE_ALPHAS), len(splits)))
    for inner_fold, (train, valid) in enumerate(splits, 1):
        pending = []
        for candidate, alpha in enumerate(RIDGE_ALPHAS):
            expected = {
                "context": context,
                "inner_fold": inner_fold,
                "method": "Ridge",
                "candidate": candidate,
                "alpha": alpha,
            }
            path = candidate_path(context, "Ridge", candidate, inner_fold)
            cached = cached_candidate(path, expected)
            if cached is None:
                pending.append((candidate, alpha, path, expected))
            else:
                scores[candidate, inner_fold - 1] = cached["mse"]
        if not pending:
            continue
        scaler = StandardScaler()
        train_features = scaler.fit_transform(X_train[train])
        valid_features = scaler.transform(X_train[valid])
        for candidate, alpha, path, expected in pending:
            started = time.perf_counter()
            model = Ridge(alpha=alpha, solver="lsqr", tol=1e-6).fit(
                train_features, y_train[train]
            )
            mse = mean_squared_error(y_train[valid], model.predict(valid_features))
            scores[candidate, inner_fold - 1] = mse
            row = {**expected, "mse": mse, "seconds": time.perf_counter() - started}
            write_json_checkpoint(path, row)
            remember_tuning(row)
        atomic_csv(pd.DataFrame(tuning_records), RUN_DIR / "tuning_progress.csv")
    means = scores.mean(axis=1)
    best = int(means.argmin())
    return RIDGE_ALPHAS[best], float(means[best])


with threadpool_limits(limits=CPU_THREADS):
    for fold, (train, test) in enumerate(outer_splits, 1):
        if restore_outer(fold, "Ridge"):
            print(f"Outer {fold}: Ridge model already complete; skipped.", flush=True)
            continue
        alpha, inner_mse = tune_ridge(
            X[train], y[train], inner_splits[fold], context=f"outer_{fold}"
        )
        started = time.perf_counter()
        ridge_model = make_pipeline(
            StandardScaler(), Ridge(alpha=alpha, solver="lsqr", tol=1e-6)
        ).fit(X[train], y[train])
        fit_seconds = time.perf_counter() - started
        prediction = ridge_model.predict(X[test])
        baseline_prediction = np.full(len(test), y[train].mean())
        records = []
        for name, values, params in [
            ("Ridge", prediction, {"alpha": alpha}),
            ("Mean baseline", baseline_prediction, {}),
        ]:
            records.append(
                {
                    "fold": fold,
                    "method": name,
                    "best_params": json.dumps(params),
                    "inner_mse": inner_mse if name == "Ridge" else np.nan,
                    "n_train": len(train),
                    "n_test": len(test),
                    "fit_seconds": fit_seconds if name == "Ridge" else 0,
                    **metrics(y[test], values),
                }
            )
        commit_outer(
            {
                "fold": fold,
                "method": "Ridge",
                "model": ridge_model,
                "test_indices": test,
                "predictions": {
                    "Ridge": prediction,
                    "Mean baseline": baseline_prediction,
                },
                "records": records,
            }
        )
        print(
            f"Outer {fold}: Ridge alpha={alpha:g}, held-out MSE="
            f"{mean_squared_error(y[test], prediction):.5f}",
            flush=True,
        )

## 7. Out-of-fold comparison and visualizations

Only held-out predictions are used below. The pooled metrics are calculated over
all images; outer-fold mean and SD are also reported. Positive MSE improvement
means better than the mean-only baseline.

Plots include paired fold errors, true versus predicted ratings, residuals, errors
in rating tails, predicted versus observed calibration, and inner-CV tuning curves.
CURE curves show the best representative/shrinkage setting **at each k using inner
validation only**. Separate tables preserve every candidate's full parameter values.
Do not repeatedly adjust grids based on these outer-test plots and then describe
the same folds as an untouched test set.

In [ ]:
if any(not np.isfinite(prediction).all() for prediction in oof_predictions.values()):
    raise RuntimeError(
        "Incomplete out-of-fold predictions: finish all folds before comparing."
    )
fold_scores = pd.DataFrame(fold_records)
fold_scores.to_csv(RUN_DIR / "fold_scores.csv", index=False)
display(
    fold_scores[
        ["fold", "method", "best_params", "inner_mse", "mse", "rmse", "mae", "r2"]
    ]
)
display(
    fold_scores.groupby("method")[["mse", "rmse", "mae", "r2"]].agg(["mean", "std"])
)
pooled = pd.DataFrame(
    [
        {"method": name, **metrics(y, prediction)}
        for name, prediction in oof_predictions.items()
    ]
).set_index("method")
pooled["mse_improvement_vs_mean"] = pooled.loc["Mean baseline", "mse"] - pooled["mse"]
pooled.to_csv(RUN_DIR / "pooled_scores.csv")
display(pooled)
oof = pd.DataFrame(
    {
        "image_id": image_ids,
        "mean_score": y,
        "outer_fold": outer_fold,
        **oof_predictions,
    }
)
oof.to_csv(RUN_DIR / "oof_predictions.csv", index=False)
tuning = pd.DataFrame(tuning_records)
tuning.to_csv(RUN_DIR / "tuning_scores.csv", index=False)
inner_summary = tuning.groupby(["context", "method", "candidate"], as_index=False).agg(
    inner_mse=("mse", "mean"), inner_sd=("mse", "std")
)
candidate_params = tuning.drop_duplicates(["context", "method", "candidate"]).drop(
    columns=["inner_fold", "mse", "seconds"]
)
inner_summary = inner_summary.merge(
    candidate_params, on=["context", "method", "candidate"]
)
inner_summary.to_csv(RUN_DIR / "inner_candidate_summary.csv", index=False)


def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(RUN_DIR / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in oof_predictions:
    rows = fold_scores[fold_scores["method"] == name]
    axes[0].plot(rows["fold"], rows["mse"], "o-", label=name)
pooled["mse"].plot.bar(ax=axes[1], rot=20)
axes[0].set(
    xlabel="Outer fold", ylabel="Held-out MSE", xticks=range(1, OUTER_FOLDS + 1)
)
axes[0].legend()
axes[1].set(ylabel="Pooled out-of-fold MSE", xlabel="", title="Same held-out images")
save_figure(fig, "model_comparison")

for name, prediction in oof_predictions.items():
    if name == "Mean baseline":
        continue
    residual = prediction - y
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    density = axes[0].hexbin(y, prediction, gridsize=45, mincnt=1, bins="log")
    axes[0].plot([1, 10], [1, 10], "k--")
    axes[0].set(xlabel="True mean rating", ylabel="Predicted rating", title=name)
    fig.colorbar(density, ax=axes[0], label="Images (log scale)")
    axes[1].hexbin(prediction, residual, gridsize=45, mincnt=1, bins="log")
    axes[1].axhline(0, color="black", linestyle="--")
    axes[1].set(
        xlabel="Predicted rating",
        ylabel="Prediction - true",
        title="Residual structure",
    )
    axes[2].hist(residual, bins=60, edgecolor="white")
    axes[2].axvline(0, color="black", linestyle="--")
    axes[2].set(
        xlabel="Prediction - true", ylabel="Images", title="Out-of-fold residuals"
    )
    save_figure(fig, name.lower().replace("-", "_") + "_diagnostics")

rating_bins = pd.cut(y, bins=np.arange(1, 11), include_lowest=True)
bin_rows = []
calibration_rows = []
for name, prediction in oof_predictions.items():
    rows = pd.DataFrame(
        {
            "bin": rating_bins,
            "error": prediction - y,
            "absolute_error": np.abs(prediction - y),
        }
    )
    grouped = rows.groupby("bin", observed=False)
    summary = grouped.agg(
        images=("error", "size"), mae=("absolute_error", "mean"), bias=("error", "mean")
    ).reset_index()
    summary["method"] = name
    bin_rows.append(summary)
    calibration = (
        pd.DataFrame(
            {
                "bin": pd.cut(prediction, bins=10, include_lowest=True),
                "true": y,
                "prediction": prediction,
            }
        )
        .groupby("bin", observed=True)
        .agg(
            images=("true", "size"),
            observed=("true", "mean"),
            predicted=("prediction", "mean"),
        )
        .reset_index()
    )
    calibration["method"] = name
    calibration_rows.append(calibration)
bin_errors = pd.concat(bin_rows, ignore_index=True)
calibration = pd.concat(calibration_rows, ignore_index=True)
bin_errors.to_csv(RUN_DIR / "rating_bin_errors.csv", index=False)
calibration.to_csv(RUN_DIR / "calibration.csv", index=False)
display(bin_errors)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in oof_predictions:
    rows = bin_errors[bin_errors["method"] == name]
    axes[0].plot(rows["bin"].astype(str), rows["mae"], "o-", label=name)
    rows = calibration[calibration["method"] == name]
    axes[1].plot(rows["predicted"], rows["observed"], "o-", label=name)
axes[0].tick_params(axis="x", rotation=45)
axes[0].set(
    xlabel="True rating bin", ylabel="Out-of-fold MAE", title="Errors in rating tails"
)
axes[1].plot([1, 10], [1, 10], "k--")
axes[1].set(
    xlabel="Mean predicted rating", ylabel="Mean observed rating", title="Calibration"
)
for ax in axes:
    ax.legend()
save_figure(fig, "tail_errors_and_calibration")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ["K-Means", "CURE", "Ridge"]):
    rows = inner_summary[inner_summary["method"] == name]
    if rows.empty:
        ax.set_title(f"{name}: not run")
        continue
    x_column = "alpha" if name == "Ridge" else "n_clusters"
    for context, group in rows.groupby("context"):
        group = group.groupby(x_column, as_index=False)["inner_mse"].min()
        ax.plot(group[x_column], group["inner_mse"], "o-", label=context)
    ax.set(xlabel=x_column, ylabel="Inner validation MSE", title=name)
    ax.set_xscale("log")
    ax.legend()
save_figure(fig, "inner_tuning_curves")

## 8. Final models (distinct from nested-CV evaluation)

Nested CV produces **one winning parameter set per outer fold**, not one globally
best model. For deployment, do a fresh 3-fold search on all available images, then
refit the selected model on all images. That final search score is a tuning score,
**not another unbiased test score**; report the nested out-of-fold results instead.

The independent final **Ridge model is saved by default**. Final K-Means and sampled
CURE models are opt-in because their full-data searches add substantial work.
The final CURE model uses the same fixed sampling seed and saves its final 10,000
sampled image IDs alongside the fitted pipeline.
No outer-fold winner is chosen using outer-test error. Saved pipelines accept raw
524-column arrays in the exact order **512 CLIP values, then 12 structural values**.
Only load trusted joblib files, and use the recorded package versions.
Completed final searches reuse their per-candidate score checkpoints. Final models
are checkpointed immediately after each full-data fit and exported as ordinary
`*_final.joblib` pipelines. On restart, a completed model is restored, not retrained.
If interruption happens during one final fit, only that fit must restart.

In [ ]:
final_params = {}
final_splits = list(
    KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED + 1000).split(X)
)
with threadpool_limits(limits=CPU_THREADS):
    if RUN_FINAL_RIDGE_MODEL:
        path = CHECKPOINT_DIR / "models" / "final_ridge.joblib"
        payload = read_model_checkpoint(path)
        if payload is None:
            alpha, inner_mse = tune_ridge(X, y, final_splits, context="final_full_data")
            final_ridge = make_pipeline(
                StandardScaler(), Ridge(alpha=alpha, solver="lsqr", tol=1e-6)
            ).fit(X, y)
            payload = {
                "config_hash": CONFIG_HASH,
                "model": final_ridge,
                "params": {"alpha": alpha, "tuning_mse": inner_mse},
            }
            atomic_model(path, payload)
            print(f"Saved final Ridge: alpha={alpha:g}")
        else:
            print("Restored final Ridge; training skipped.")
        if not (RUN_DIR / "ridge_final.joblib").is_file():
            atomic_model(RUN_DIR / "ridge_final.joblib", payload["model"])
        final_params["Ridge"] = payload["params"]
    if RUN_FINAL_CLUSTER_MODELS:
        cached_models = {
            name: read_model_checkpoint(
                CHECKPOINT_DIR / "models" / f"final_{method_slug(name)}.joblib"
            )
            for name in cluster_grids
        }
        pending_methods = [
            name for name, cached in cached_models.items() if cached is None
        ]
        if pending_methods:
            selected = tune_clusters(
                X,
                y,
                final_splits,
                SEED + 1000,
                context="final_full_data",
                methods=pending_methods,
            )
            preprocessor = cluster_preprocessor(SEED + 1000)
            features = preprocessor.fit_transform(X)
            for name in pending_methods:
                method = "kmeans" if name == "K-Means" else "cure"
                params, inner_mse = selected[name]
                model = cluster_estimator(method, params, SEED + 1000).fit(features, y)
                payload = {
                    "config_hash": CONFIG_HASH,
                    "model": make_pipeline(preprocessor, model),
                    "params": {**params, "tuning_mse": inner_mse},
                }
                if name == "CURE":
                    payload["sample_data"] = {
                        "image_id": image_ids[model.sample_indices_],
                        "dataset_row": model.sample_indices_,
                        "sample_seed": np.full(model.sample_size_, model.sample_seed),
                    }
                atomic_model(
                    CHECKPOINT_DIR / "models" / f"final_{method_slug(name)}.joblib",
                    payload,
                )
                cached_models[name] = payload
                print(f"Saved final {name}: {params}")
            del features, preprocessor, model
        for name, payload in cached_models.items():
            method = "kmeans" if name == "K-Means" else "cure"
            if not (RUN_DIR / f"{method}_final.joblib").is_file():
                atomic_model(RUN_DIR / f"{method}_final.joblib", payload["model"])
            if name == "CURE":
                atomic_csv(
                    pd.DataFrame(payload["sample_data"]),
                    RUN_DIR / "cure_selected_sample_final.csv",
                )
            final_params[name] = payload["params"]
            if name not in pending_methods:
                print(f"Restored final {name}; training skipped.")
atomic_json(RUN_DIR / "final_params.json", final_params)
atomic_csv(pd.DataFrame(tuning_records), RUN_DIR / "tuning_scores.csv")
print(f"All artifacts: {RUN_DIR}")
print(
    "For a saved cluster pipeline in another notebook, run the estimator-definition cell "
    "before joblib.load. Ridge requires only the recorded scikit-learn dependencies."
)

## 9. Optional: full-feature K-Means (no PCA), independently resumable

**Copy this section into the end of your existing notebook.** Run it after the
setup cells (sections 1-3); do not rerun sections 4-8. If the current runtime still
has `X`, `y`, `outer_splits`, and `inner_splits`, just run this new cell.

This keeps **all 524 standardized features**, matching Ridge's input
representation, and selects k with the same 3x3 splits. It does not change
`PCA_COMPONENTS`, existing grids, results, or models. All configured `K_VALUES` are
retained for a controlled comparison, so expect a more expensive search than
20-component K-Means. No full-data production model is fitted in this section.

Each inner candidate score and each outer model/prediction is atomically saved
under its own `kmeans_full_features` subfolder. Rerunning skips completed work.
Keep the original `RUN_NAME`, data, and settings unchanged to resume. Checkpoint
compatibility includes the actual arrays and split indices, not just row counts.
Only load trusted model files. Do not write to one run from multiple sessions.

The original pooled-score CSV is read, if available, for a side-by-side comparison;
no Ridge/CURE rerun is needed. This is a follow-up representation experiment:
because the original outer results motivated it, it is not a new independent test.

In [ ]:
def run_full_feature_kmeans():
    name = "K-Means (524 features, no PCA)"
    directory = RUN_DIR / "kmeans_full_features"
    fingerprint = hashlib.sha256()
    fingerprint.update(memoryview(np.ascontiguousarray(X)))
    fingerprint.update(memoryview(np.ascontiguousarray(y)))
    fingerprint.update(json.dumps(image_ids.tolist()).encode())
    for fold, (train, test) in enumerate(outer_splits, 1):
        for indices in (train, test):
            fingerprint.update(memoryview(np.asarray(indices, dtype=np.int64)))
        for inner_train, inner_valid in inner_splits[fold]:
            for indices in (inner_train, inner_valid):
                fingerprint.update(memoryview(np.asarray(indices, dtype=np.int64)))
    settings = {
        "version": 1,
        "parent_config": CONFIG_HASH,
        "data_and_splits_sha256": fingerprint.hexdigest(),
        "k_values": K_VALUES,
        "n_init": KMEANS_N_INIT,
        "seed": SEED,
        "n_features": X.shape[1],
        "preprocessing": "StandardScaler, no PCA",
    }
    manifest = directory / "config.json"
    if manifest.is_file():
        if json.loads(manifest.read_text()) != settings:
            raise ValueError(
                "Full-feature settings/data changed. Use a new run folder."
            )
    else:
        if directory.exists() and any(directory.iterdir()):
            raise ValueError("Full-feature folder has files but no configuration.")
        atomic_json(manifest, settings)
    if max(K_VALUES) > min(
        len(train) for splits in inner_splits.values() for train, _ in splits
    ):
        raise ValueError("Too many clusters for an inner training fold.")
    prediction = np.full(len(y), np.nan)
    records, tuning = [], []
    with threadpool_limits(limits=CPU_THREADS):
        for fold, (train, test) in enumerate(outer_splits, 1):
            score_matrix = np.empty((len(K_VALUES), len(inner_splits[fold])))
            pending = []
            for inner, (fit_rows, valid_rows) in enumerate(inner_splits[fold], 1):
                missing = []
                for candidate, k in enumerate(K_VALUES):
                    score_path = directory / f"outer_{fold}_inner_{inner}_k_{k}.json"
                    if score_path.is_file():
                        row = json.loads(score_path.read_text())
                        if (row["fold"], row["inner"], row["k"]) != (fold, inner, k):
                            raise ValueError(
                                f"Wrong candidate checkpoint: {score_path}"
                            )
                        if not np.isfinite(row["mse"]):
                            raise ValueError(f"Non-finite MSE: {score_path}")
                        score_matrix[candidate, inner - 1] = row["mse"]
                        tuning.append(row)
                    else:
                        missing.append((candidate, k, score_path))
                if missing:
                    pending.append((inner, fit_rows, valid_rows, missing))
            for inner, fit_rows, valid_rows, missing in pending:
                scaler = StandardScaler()
                fit_features = scaler.fit_transform(X[train[fit_rows]])
                valid_features = scaler.transform(X[train[valid_rows]])
                for candidate, k, score_path in missing:
                    started = time.perf_counter()
                    model = ClusterMeanRegressor(
                        method="kmeans",
                        n_clusters=k,
                        n_init=KMEANS_N_INIT,
                        random_state=SEED + fold * 100 + inner,
                    ).fit(fit_features, y[train[fit_rows]])
                    mse = mean_squared_error(
                        y[train[valid_rows]], model.predict(valid_features)
                    )
                    row = {
                        "fold": fold,
                        "inner": inner,
                        "k": k,
                        "mse": mse,
                        "seconds": time.perf_counter() - started,
                    }
                    atomic_json(score_path, row)
                    score_matrix[candidate, inner - 1] = mse
                    tuning.append(row)
                    print(
                        f"Full-feature K-Means: outer {fold}, inner {inner}, "
                        f"k={k}, MSE={mse:.5f} [saved]",
                        flush=True,
                    )
                del fit_features, valid_features, model
            means = score_matrix.mean(axis=1)
            best_index = int(means.argmin())
            best_k = K_VALUES[best_index]
            model_path = directory / f"outer_{fold}_model.joblib"
            if model_path.is_file():
                payload = joblib.load(model_path)
                if payload["best_k"] != best_k or not np.array_equal(
                    payload["test_indices"], test
                ):
                    raise ValueError(
                        "Outer model does not match the selected k/test fold."
                    )
                print(f"Full-feature K-Means: outer {fold} restored.", flush=True)
            else:
                started = time.perf_counter()
                model = make_pipeline(
                    StandardScaler(),
                    ClusterMeanRegressor(
                        method="kmeans",
                        n_clusters=best_k,
                        n_init=KMEANS_N_INIT,
                        random_state=SEED + fold * 100,
                    ),
                ).fit(X[train], y[train])
                values = model.predict(X[test])
                payload = {
                    "model": model,
                    "test_indices": test,
                    "best_k": best_k,
                    "prediction": values,
                    "record": {
                        "fold": fold,
                        "method": name,
                        "best_k": best_k,
                        "inner_mse": float(means[best_index]),
                        "fit_seconds": time.perf_counter() - started,
                        **metrics(y[test], values),
                    },
                }
                atomic_model(model_path, payload)
            values = payload["prediction"]
            if values.shape != (len(test),) or not np.isfinite(values).all():
                raise ValueError("Invalid saved outer predictions.")
            prediction[test] = values
            records.append(payload["record"])
            atomic_csv(pd.DataFrame(records), directory / "fold_scores.csv")
            atomic_csv(
                pd.DataFrame(
                    {
                        "image_id": image_ids,
                        "mean_score": y,
                        "prediction": prediction,
                    }
                ),
                directory / "oof_predictions.csv",
            )
    atomic_csv(pd.DataFrame(tuning), directory / "inner_scores.csv")
    result = pd.DataFrame([{"method": name, **metrics(y, prediction)}]).set_index(
        "method"
    )
    atomic_csv(result, directory / "pooled_scores.csv", index=True)
    display(pd.DataFrame(records))
    original_scores = RUN_DIR / "pooled_scores.csv"
    comparison = (
        pd.concat([pd.read_csv(original_scores, index_col=0), result], sort=False)
        if original_scores.is_file()
        else result
    )
    display(comparison[["mse", "rmse", "mae", "r2"]])
    fig, ax = plt.subplots(figsize=(9, 4))
    comparison["mse"].plot.bar(ax=ax, rot=25)
    ax.set(
        ylabel="Out-of-fold MSE",
        xlabel="",
        title="Same image folds; different representations",
    )
    fig.tight_layout()
    fig.savefig(directory / "comparison.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Full-feature experiment saved to:", directory)
    return prediction, pd.DataFrame(records), comparison


full_feature_predictions, full_feature_folds, full_feature_comparison = (
    run_full_feature_kmeans()
)